In [124]:
# Multi-dimensional arrays and mathematical functions
import numpy as np

# Tabular data manipulation and analysis (DataFrames)
import pandas as pd

# Splitting data into training and testing sets
from sklearn.model_selection import train_test_split

# Regular expressions for advanced text search and manipulation
import re

# Natural Language Toolkit for text processing
import nltk

# Built-in string operations (like list of punctuation)
import string

# List of common words to filter out (e.g., "the", "is", "at")
from nltk.corpus import stopwords

# Text normalization tools: reducing words to their base or root forms
from nltk.stem import SnowballStemmer, WordNetLemmatizer

# Converts a collection of text documents into a matrix of token counts
from sklearn.feature_extraction.text import CountVectorizer

# Optimized gradient boosting library for high-performance modeling
import xgboost as xgb

# Evaluation metric: ratio of correctly predicted observations
from sklearn.metrics import accuracy_score

# Evaluation metrics: tracking false positives, false negatives, and overall model confidence
from sklearn.metrics import precision_score, recall_score, roc_auc_score


### Data Ingestion

In [125]:
df = pd.read_csv('https://raw.githubusercontent.com/Donatus-Victor/ML-Pipeline-with-DVC/refs/heads/main/tweet_emotions.csv')

df.head()

,tweet_id,sentiment,content
0,1956967341,empty,@tiffanylue i know i was listenin to bad habi...
1,1956967666,sadness,Layin n bed with a headache ughhhh...waitin o...
2,1956967696,sadness,Funeral ceremony...gloomy friday...
3,1956967789,enthusiasm,wants to hang out with friends SOON!
4,1956968416,neutral,@dannycastillo We want to trade with someone w...


In [126]:
# delete tweet id
df.drop(columns=['tweet_id'],inplace=True)
df

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...
...,...,...
39995,neutral,@JohnLloydTaylor
39996,love,Happy Mothers Day All my love
39997,love,Happy Mother's Day to all the mommies out ther...
39998,happiness,@niariley WASSUP BEAUTIFUL!!! FOLLOW ME!! PEE...


In [127]:
final_df = df[df['sentiment'].isin(['happiness','sadness'])]

In [128]:
final_df.sample(5)

,sentiment,content
21956,sadness,@Gjerninger Are you sure you want to know? It...
28152,happiness,there. She says it gives her an excuse to sle...
3787,sadness,I miss my wacom. Especially the mouse. Laptop ...
34845,happiness,@dmcox fantastic day in the AZ sun
36261,happiness,What a fantastic full-on weekend! Going to fi...


In [129]:
final_df.shape

(10374, 2)

In [130]:
final_df['sentiment'].replace({'happiness':1, 'sadness':0},inplace=True)

C:\Users\DONATUS\AppData\Local\Temp\ipykernel_9076\4156243374.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  final_df['sentiment'].replace({'happiness':1, 'sadness':0},inplace=True)


1        0
2        0
6        0
8        0
9        0
        ..
39986    1
39987    1
39988    1
39994    1
39998    1
Name: sentiment, Length: 10374, dtype: object

In [131]:
final_df.head()

,sentiment,content
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
6,sadness,"I should be sleep, but im not! thinking about ..."
8,sadness,@charviray Charlene my love. I miss you
9,sadness,@kelcouch I'm sorry at least it's Friday?


In [132]:
train_data, test_data = train_test_split(final_df, test_size=0.2, random_state=42)

### Data Preprocessing

In [133]:
# Download required datasets for language processing (words definitions and common filler words)
nltk.download('wordnet')
nltk.download('stopwords')

# Reduce words to their base/root form (e.g., "running" becomes "run")
def lemmatization(text):
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(y) for y in text]
    return " " .join(text)

# Remove common words that carry little meaning (like "the", "is", "at")
def remove_stop_words(text):
    stop_words = set(stopwords.words("english"))
    Text = [i for i in str(text).split() if i not in stop_words]
    return " ".join(Text)

# Remove all numbers from the text
def removing_numbers(text):
    text = ''.join([i for i in text if not i.isdigit()])
    return text

# Convert all text to lowercase so capitalization doesn't confuse the model
def lower_case(text):
    text = text.split()
    text = [y.lower() for y in text]
    return " " .join(text)

# Remove punctuation marks and clean up any accidental extra spaces
def removing_punctuations(text):
    text = re.sub('[%s]' % re.escape("""!"#$%&'()*+,،-./:;<=>؟?@[\]^_`{|}~"""), ' ', text)
    text = text.replace('؛',"", )
    text = re.sub('\s+', ' ', text)
    text = " ".join(text.split())
    return text.strip()

# Remove website links (URLs) from the text
def removing_urls(text):
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

# Delete rows where the text is too short (fewer than 3 words)
def remove_small_sentences(df):
    for i in range(len(df)):
        if len(df.text.iloc[i].split()) < 3:
            df.text.iloc[i] = np.nan

# Clean and normalize an entire column of text in a dataset using all rules above
def normalize_text(df):
    df.content = df.content.apply(lambda content : lower_case(content))
    df.content = df.content.apply(lambda content : remove_stop_words(content))
    df.content = df.content.apply(lambda content : removing_numbers(content))
    df.content = df.content.apply(lambda content : removing_punctuations(content))
    df.content = df.content.apply(lambda content : removing_urls(content))
    df.content = df.content.apply(lambda content : lemmatization(content))
    return df

# Clean and normalize a single text sentence using all rules above
def normalized_sentence(sentence):
    sentence = lower_case(sentence)
    sentence = remove_stop_words(sentence)
    sentence = removing_numbers(sentence)
    sentence = removing_punctuations(sentence)
    sentence = removing_urls(sentence)
    sentence = lemmatization(sentence)
    return sentence

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\DONATUS\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DONATUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [134]:
normalized_sentence("That's it? It's done already? This is one")

'that s it done already one'

In [135]:
train_data = normalize_text(train_data)
test_data = normalize_text(test_data)

In [136]:
train_data

,sentiment,content
23531,sadness,quot my problem miss you cause don t quot
8051,sadness,that s it done already one proof there s nothi...
11499,sadness,hungry food steal
31288,happiness,foot hurt finally bed will forget crunch over ...
18561,sadness,really ill atm
...,...,...
21697,happiness,chocolatesuze yes yes should especially wine m...
19445,sadness,kickzfadayz boy better get tonight
20216,happiness,tafe actually quite good
3258,sadness,minute boarding hour home window seat


### feature engineering

In [137]:
X_train = train_data['content'].values
y_train = train_data['sentiment'].values

X_test = test_data['content'].values
y_test = test_data['sentiment'].values

In [138]:
# Apply Bag of Words (CountVectorizer)
vectorizer = CountVectorizer()

# Fit the vectorizer on the training data and transform it
X_train_bow = vectorizer.fit_transform(X_train)

# Transform the test data using the same vectorizer
X_test_bow = vectorizer.transform(X_test)

In [139]:
train_df = pd.DataFrame(X_train_bow.toarray())

train_df['label'] = y_train
train_df.head()

,0,1,2,3,4,5,6,7,8,9,...,14223,14224,14225,14226,14227,14228,14229,14230,14231,label
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,sadness
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,sadness
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,sadness
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,happiness
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,sadness


#### Model building

In [140]:
# Import the missing evaluation tool
from sklearn.metrics import classification_report, accuracy_score
import xgboost as xgb

# 1. Overwrite the text labels with the 0 and 1 values
y_train = y_train.map({'happiness': 1, 'sadness': 0})
y_test = y_test.map({'happiness': 1, 'sadness': 0})

# 2. Define and train the XGBoost model (Removed the deprecated use_label_encoder)
xgb_model = xgb.XGBClassifier(eval_metric='mlogloss')
xgb_model.fit(X_train_bow, y_train)

# 3. Make predictions
y_pred = xgb_model.predict(X_test_bow)

# 4. Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
classification_rep = classification_report(y_test, y_pred)

print("Accuracy:", accuracy)
print("Classification Report:\n", classification_rep)


Accuracy: 0.771566265060241
Classification Report:
               precision    recall  f1-score   support

           0       0.75      0.83      0.79      1060
           1       0.80      0.71      0.75      1015

    accuracy                           0.77      2075
   macro avg       0.77      0.77      0.77      2075
weighted avg       0.77      0.77      0.77      2075



In [141]:
# # Define and train the XGBoost model
# xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
# xgb_model.fit(X_train_bow, y_train)

# # Make predictions
# y_pred = xgb_model.predict(X_test_bow)

# # Evaluate the model
# accuracy = accuracy_score(y_test, y_pred)
# classification_rep = classification_report(y_test, y_pred)

# print("Accuracy:", accuracy)
# print("Classification Report:\n", classification_rep)

#### Model evaluation

In [142]:
# Make predictions
y_pred = xgb_model.predict(X_test_bow)
y_pred_proba = xgb_model.predict_proba(X_test_bow)[:, 1]

# Calculate evaluation metrics
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

In [143]:
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"AUC: {auc}")

Precision: 0.7988950276243094
Recall: 0.7123152709359606
AUC: 0.8595775629705363
